# Field Scan : DeepForest en validation croisée par ferme

Les 5 fermes hors test servent chacune **une fois** de validation. À chaque tour, on **repart du modèle DeepForest
d'origine** (« arbres »), on l'entraîne sur les 4 autres fermes, puis on le note sur la ferme laissée de côté.
La ferme de test (**Nestor Macias**) n'est jamais utilisée ici.

Règle de notation du sujet : une détection est juste si son centre est à moins de 1 m du centre d'un vrai bananier.
Pour la ferme notée, on ne garde que ses tuiles **sans chevauchement**, pour ne compter chaque bananier qu'une fois.

**Reprise après coupure** : chaque tour terminé est enregistré sur le Drive ; si Colab se déconnecte, il suffit de
tout relancer, les tours déjà faits sont sautés.

Avant de lancer : **GPU T4**. Données : `MyDrive/field_scan/tuiles.zip`. Durée : environ 1 h à 1 h 30.

In [ ]:
# 1. Installation, Drive, tuiles
!pip install -q "deepforest>=2,<3"
from google.colab import drive
drive.mount('/content/drive')
DOSSIER_DRIVE = '/content/drive/MyDrive/field_scan'
!unzip -q -o "{DOSSIER_DRIVE}/tuiles.zip" -d /content/

In [ ]:
# 2. Réglages
import json, time, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
import deepforest
from deepforest import main
from scipy.optimize import linear_sum_assignment

RACINE = Path('/content/tuiles')
IMAGES = RACINE / 'images'
SORTIE = Path(DOSSIER_DRIVE) / 'resultats_comptage' / 'validation_croisee_deepforest'
SORTIE.mkdir(parents=True, exist_ok=True)
FERMES = ['Flora Pluas RGB', 'Leonor Aspiazu RGB', 'Carlos Vera Guevara RGB',
          'Carlos Vera Arteaga RGB', 'Manuel Macias RGB']          # Nestor Macias = test, exclu
EPOQUES = 10
TAILLE_LOT = 8
RAYON_M = 1.0
SEUILS_CONFIANCE = np.round(np.arange(0.05, 0.96, 0.05), 2)
SEUILS_NMS = [0.05, 0.2, 0.4]
LABEL = 'Tree'
print('DeepForest', deepforest.__version__, '| GPU :', torch.cuda.is_available())

tuiles = pd.read_csv(RACINE / 'tuiles.csv')
annotations = pd.read_csv(RACINE / 'annotations.csv')
# Position de la tuile dans l'image, lue dans son nom : <ferme>_<ligne>_<colonne>.png
tuiles['y0'] = tuiles.image_path.str.extract(r'_(\d+)_\d+\.png$')[0].astype(int)
tuiles['x0'] = tuiles.image_path.str.extract(r'_(\d+)\.png$')[0].astype(int)
TAILLE_TUILE = 400

In [ ]:
# 3. Outils : fichier d'apprentissage, notation à 1 m
def fichier_apprentissage(fermes, chemin):
    noms = tuiles[tuiles.site.isin(fermes)].image_path
    bananes = annotations[(annotations.label == 'banane') & annotations.image_path.isin(noms)].copy()
    bananes = bananes[(bananes.xmax > bananes.xmin) & (bananes.ymax > bananes.ymin)]
    bananes['label'] = LABEL
    vides = pd.DataFrame({'image_path': sorted(set(noms) - set(bananes.image_path)),
                          'xmin': 0, 'ymin': 0, 'xmax': 0, 'ymax': 0, 'label': LABEL})
    pd.concat([bananes[['image_path', 'xmin', 'ymin', 'xmax', 'ymax', 'label']], vides]).to_csv(chemin, index=False)
    return chemin

def tuiles_de_validation(ferme):
    # Tuiles sans chevauchement : celles dont la position est un multiple de la taille de tuile
    t = tuiles[tuiles.site == ferme]
    return t[(t.y0 % TAILLE_TUILE == 0) & (t.x0 % TAILLE_TUILE == 0)]

def centres(df):
    return np.c_[(df.xmin + df.xmax) / 2, (df.ymin + df.ymax) / 2]

def apparier(det, vrai, rayon_px):
    if len(det) == 0 or len(vrai) == 0:
        return 0
    d = np.linalg.norm(det[:, None] - vrai[None], axis=2)
    cout = np.where(d <= rayon_px, d, 1e6)
    l, c = linear_sum_assignment(cout)
    return int((cout[l, c] <= rayon_px).sum())

def grille_de_notation(modele, validation):
    verite = annotations[(annotations.label == 'banane') & annotations.image_path.isin(validation.image_path)]
    lignes = []
    for nms in SEUILS_NMS:
        modele.model.nms_thresh = nms
        modele.model.score_thresh = 0.05
        pred = {}
        for nom in validation.image_path:
            p = modele.predict_image(path=str(IMAGES / nom))
            pred[nom] = p if p is not None else pd.DataFrame(columns=['xmin', 'ymin', 'xmax', 'ymax', 'score'])
        for s in SEUILS_CONFIANCE:
            vp = nb_det = 0
            for _, t in validation.iterrows():
                p = pred[t.image_path]; p = p[p.score >= s]
                vp += apparier(centres(p), centres(verite[verite.image_path == t.image_path]), RAYON_M / t.taille_pixel_m)
                nb_det += len(p)
            lignes.append({'nms': nms, 'seuil': s, 'detections': nb_det, 'bananes_vraies': len(verite), 'vp': vp})
    return pd.DataFrame(lignes)

In [ ]:
# 4. Les 5 tours (un modèle neuf à chaque tour)
try:
    from lightning.pytorch.loggers import CSVLogger
except ImportError:
    from pytorch_lightning.loggers import CSVLogger

for ferme in FERMES:
    nom = ferme.replace(' RGB', '').replace(' ', '_')
    if (SORTIE / f'grille_{nom}.csv').exists():
        print(f'{ferme} : déjà fait, tour sauté')
        continue
    apprentissage = [f for f in FERMES if f != ferme]
    validation = tuiles_de_validation(ferme)
    print(f'\n=== Validation sur {ferme} ({len(validation)} tuiles) ; apprentissage sur {apprentissage}')

    csv_app = fichier_apprentissage(apprentissage, RACINE / f'app_{nom}.csv')
    torch.manual_seed(0)
    modele = main.deepforest()                          # modèle d'origine, neuf à chaque tour
    modele.load_model('weecology/deepforest-tree')
    modele.config.train.csv_file = str(csv_app)
    modele.config.train.root_dir = str(IMAGES)
    modele.config.train.epochs = EPOQUES
    modele.config.batch_size = TAILLE_LOT
    modele.config.workers = 2
    journal = CSVLogger('/content/journaux', name=nom)
    debut = time.time()
    modele.create_trainer(logger=journal)
    modele.trainer.fit(modele)
    print(f'Apprentissage : {(time.time() - debut) / 60:.1f} min')
    # Sécurité : si l'apprentissage s'est arrêté avant la fin (coupure, erreur), on n'enregistre pas ce tour
    epoques_faites = pd.read_csv(Path(journal.log_dir) / 'metrics.csv').epoch.max() + 1
    if epoques_faites < EPOQUES:
        raise RuntimeError(f'{ferme} : seulement {epoques_faites} époques sur {EPOQUES}, tour non enregistré. Relancer.')

    grille = grille_de_notation(modele, validation)
    grille.insert(0, 'ferme', ferme)
    grille.to_csv(SORTIE / f'grille_{nom}.csv', index=False)
    torch.save(modele.model.state_dict(), SORTIE / f'poids_sans_{nom}.pt')
    pd.read_csv(Path(journal.log_dir) / 'metrics.csv').to_csv(SORTIE / f'journal_{nom}.csv', index=False)
    print(f"Meilleure F1 sur {ferme} : {grille.assign(f1=2 * grille.vp / (grille.detections + grille.bananes_vraies)).f1.max():.3f}")
    del modele; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 5. Bilan : un réglage commun (nms, seuil), choisi sur l'ensemble des 5 fermes, puis note de chaque ferme
grilles = pd.concat([pd.read_csv(f) for f in sorted(SORTIE.glob('grille_*.csv'))])
total = grilles.groupby(['nms', 'seuil'])[['detections', 'bananes_vraies', 'vp']].sum()
total['f1'] = 2 * total.vp / (total.detections + total.bananes_vraies)
nms, seuil = total.f1.idxmax()
bilan = grilles[(grilles.nms == nms) & (grilles.seuil == seuil)].copy()
bilan['precision'] = bilan.vp / bilan.detections.clip(lower=1)
bilan['rappel'] = bilan.vp / bilan.bananes_vraies
bilan['f1'] = 2 * bilan.vp / (bilan.detections + bilan.bananes_vraies)
bilan['erreur_comptage'] = (bilan.detections - bilan.bananes_vraies) / bilan.bananes_vraies
baseline = {'Flora Pluas RGB': 0.410, 'Leonor Aspiazu RGB': 0.232, 'Carlos Vera Guevara RGB': 0.165,
            'Carlos Vera Arteaga RGB': 0.119, 'Manuel Macias RGB': 0.204}
bilan['f1_methode_simple'] = bilan.ferme.map(baseline)
pd.set_option('display.width', 200)
print(f'Réglage commun retenu : nms {nms}, seuil {seuil}\n')
print(bilan[['ferme', 'bananes_vraies', 'detections', 'vp', 'precision', 'rappel', 'f1', 'erreur_comptage',
             'f1_methode_simple']].round(3).to_string(index=False))
print(f"\nF1 globale DeepForest : {total.f1.max():.3f}   |   méthode simple : 0.247")
bilan.to_csv(SORTIE / 'bilan.csv', index=False)